<a href="https://colab.research.google.com/github/toecm/iedi-mas/blob/main/IEDI_M%C2%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""IEDI-M²: Final Integrated Version
(Strategy: Flash-First Default -> Pro-Boost for Deep Analysis)
"""

# --- INSTALL DEPENDENCIES ---
!pip install -q openai-whisper rapidfuzz pandas gradio datasets transformers torchaudio torch librosa pydub ffmpeg-python jiwer google-generativeai python-dotenv requests yt-dlp soundfile

import os
import glob
import torch
import whisper
import pandas as pd
import requests
import tempfile
import yt_dlp
import random
import soundfile as sf
import shutil
from pydub import AudioSegment
from rapidfuzz import process, fuzz
import google.generativeai as genai
from google.api_core import exceptions as google_exceptions
from datasets import load_dataset, Audio
import gradio as gr
from dotenv import load_dotenv
from threading import Lock
from huggingface_hub import HfApi, hf_hub_download, upload_file
import json
import re
import traceback
import time

# --- CONFIGURATION ---
HF_REPO_ID = "toecm/IEDID"

load_dotenv()

# Try Loading Keys from Colab Secrets
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') or os.getenv("HF_TOKEN")
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY') or os.getenv("GOOGLE_API_KEY")
    os.environ["PINATA_JWT"] = userdata.get('PINATA_JWT') or os.getenv("PINATA_JWT")
except (ImportError, Exception):
    pass

HF_TOKEN = os.getenv("HF_TOKEN")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
PINATA_JWT = os.getenv("PINATA_JWT")
DATASET_DIR = "/content/iuuy_datasets"
os.makedirs(DATASET_DIR, exist_ok=True)

# --- DYNAMIC MODEL MANAGER (Flash-First -> Pro-Boost) ---
class GeminiManager:
    def __init__(self, api_key):
        self.api_key = api_key
        if self.api_key:
            genai.configure(api_key=self.api_key)

        self.model_pro = genai.GenerativeModel("gemini-1.5-pro")
        self.model_flash = genai.GenerativeModel("gemini-1.5-flash")

        # Track which model was used last for the UI
        self.last_used_model = "Idle"
        print("🧠 Gemini Manager Online: Flash-First Mode with Pro-Boost.")

    def generate_fast(self, prompt):
        """Always uses Flash. Good for simple tasks like Regex."""
        if not self.api_key: raise Exception("Google API Key not found.")
        self.last_used_model = "gemini-1.5-flash (Fast)"
        return self.model_flash.generate_content(prompt)

    def generate_smart(self, prompt):
        """Attempts Pro first (for deep analysis). If Quota hit, falls back to Flash."""
        if not self.api_key: raise Exception("Google API Key not found.")

        try:
            # 1. Try Upgrade to Pro
            self.last_used_model = "gemini-1.5-pro (Boost)"
            return self.model_pro.generate_content(prompt)

        except Exception as e:
            # 2. Fallback to Flash on ANY error (Quota, 429, Service Unavailable)
            print(f"⚠️ Pro-Boost Failed ({str(e)[:50]}...). Falling back to Flash.")
            self.last_used_model = "gemini-1.5-flash (Fallback)"
            time.sleep(1) # Safety cooldown
            return self.model_flash.generate_content(prompt)

    def get_status_string(self):
        icon = "🚀" if "pro" in self.last_used_model else "⚡"
        return f"{icon} Last Action: {self.last_used_model}"

gemini_manager = GeminiManager(GOOGLE_API_KEY) if GOOGLE_API_KEY else None

# --- HUGGING FACE SYNC MANAGER ---
class HFManager:
    def __init__(self):
        self.api = HfApi(token=HF_TOKEN)
        self.lock = Lock()

    def pull_datasets(self):
        print("⬇️ Pulling datasets from Hugging Face...")
        try:
            files = self.api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset")
            csv_files = [f for f in files if f.endswith(".csv")]
            if not csv_files:
                seed_initial_data()
                return
            for file in csv_files:
                hf_hub_download(repo_id=HF_REPO_ID, filename=file, repo_type="dataset", local_dir=DATASET_DIR, token=HF_TOKEN)
        except Exception as e:
            print(f"❌ HF Pull Error: {e}")
            seed_initial_data()

    def push_update(self, filepath, commit_msg="Update from IEDI-MAS"):
        filename = os.path.basename(filepath)
        print(f"⬆️ Pushing update: {filename}...")
        try:
            self.api.upload_file(path_or_fileobj=filepath, path_in_repo=filename, repo_id=HF_REPO_ID, repo_type="dataset", commit_message=commit_msg)
            print("✅ Sync Complete!")
        except Exception as e: print(f"❌ HF Push Error: {e}")

    def upload_audio_sample(self, audio_path, dialect):
        clean_dialect = dialect.strip()
        filename = os.path.basename(audio_path)
        hf_path = f"audio/{clean_dialect}/{filename}"
        try:
            self.api.upload_file(path_or_fileobj=audio_path, path_in_repo=hf_path, repo_id=HF_REPO_ID, repo_type="dataset", commit_message=f"Add audio sample for {clean_dialect}")
            return hf_path
        except Exception as e:
            print(f"❌ Audio Upload Error: {e}")
            return None

hf_manager = HFManager()

def seed_initial_data():
    initial_data = {
        "Nigerian English": [{
            "Utterance": "How far?",
            "Clarification": "How are you doing?",
            "Tone_Category": "Casual/Greeting",
            "Linguistic_Context": "Common pidgin greeting functioning like 'What's up?'",
            "Syntax_Pattern": r"\bhow\s?far\b",
            "Pragmatic_Analysis": "A phatic communion greeting that expects a reciprocal inquiry rather than a literal distance measurement.",
            "file_name": ""
        }]
    }
    for dialect, rows in initial_data.items():
        filepath = os.path.join(DATASET_DIR, f"{dialect}.csv")
        if not os.path.exists(filepath):
            df = pd.DataFrame(rows)
            df["Dialect"] = dialect
            df.to_csv(filepath, index=False)
            hf_manager.push_update(filepath, "Initial Seed")

hf_manager.pull_datasets()

# --- AGENT 1: INPUT (Whisper) ---
class AgentInput:
    def __init__(self, model_size="small"):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"👂 Agent 1 (Input) Online: Loading Whisper ({model_size}) on {device}...")
        self.model = whisper.load_model(model_size, device=device)

    def transcribe(self, audio_path, language="en"):
        if not audio_path: return []
        result = self.model.transcribe(audio_path, language=language)
        return [{"speaker": "Speaker", "text": seg["text"].strip(), "start": seg["start"], "end": seg["end"]} for seg in result["segments"]]

# --- AGENT 2: INTERPRETATION (Uses Pro-Boost) ---
class AgentInterpretation:
    def __init__(self, gemini_manager_instance=None):
        self.df = pd.DataFrame()
        self.lookup_list = []
        self.gemini_manager = gemini_manager_instance
        self.lab_profile = self.load_lab_profile()
        print("🧠 Agent 2 (Interpretation) Online: Loading Datasets & Lab Context...")
        self.refresh_knowledge_base()

    def load_lab_profile(self):
        profile_path = "nsl_lab_profile.json"
        default_profile = {"lab_name": "General Context", "jargon": {}, "pragmatic_rules": []}
        if os.path.exists(profile_path):
            try:
                with open(profile_path, 'r', encoding='utf-8') as f: return json.load(f)
            except: return default_profile
        return default_profile

    def save_lab_profile(self, json_str):
        try:
            new_profile = json.loads(json_str)
            with open("nsl_lab_profile.json", "w", encoding="utf-8") as f: json.dump(new_profile, f, indent=2)
            self.lab_profile = new_profile
            self.refresh_knowledge_base()
            return "✅ Profile updated!"
        except: return "❌ Invalid JSON."

    def get_profile_text(self): return json.dumps(self.lab_profile, indent=2)

    def refresh_knowledge_base(self):
        all_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
        df_list = []
        for filename in all_files:
            try:
                dialect_name = os.path.basename(filename).replace(".csv", "")
                temp_df = pd.read_csv(filename)
                temp_df["Dialect"] = dialect_name
                df_list.append(temp_df)
            except Exception as e: print(f"⚠️ Error loading {filename}: {e}")

        if df_list:
            self.df = pd.concat(df_list, ignore_index=True)
            self.lookup_list = self.df["Utterance"].tolist()
        else:
            self.lookup_list = []

        if self.lab_profile and "jargon" in self.lab_profile:
            jargon_keys = list(self.lab_profile["jargon"].keys())
            self.lookup_list.extend(jargon_keys)

    # --- HELPER: Analyze Single Pragmatics (Uses Pro-Boost) ---
    def generate_single_pragmatics(self, text, dialect, tone):
        if not self.gemini_manager: return "LLM Offline"
        prompt = f"Briefly analyze the pragmatic intent of: '{text}' (Dialect: {dialect}, Tone: {tone}). One sentence only."
        try:
            # UPGRADE: Use smart_generate for analysis
            return self.gemini_manager.generate_smart(prompt).text.strip()
        except Exception as e: return f"Analysis Failed: {str(e)[:20]}"

    # --- FALLBACK GENERATOR (Uses Pro-Boost) ---
    def generate_unknown_analysis(self, text):
        if not self.gemini_manager: return []

        prompt = f"""
        Analyze this utterance: "{text}"
        It was NOT found in the user's known database.

        Task:
        1. Hypothesize Dialect.
        2. Provide 3 interpretations.

        Output Strictly JSON:
        [
            {{ "dialect": "Guess 1", "clarification": "Meaning 1", "tone": "Tone 1", "context": "Context 1", "pragmatics": "Intent 1" }},
            {{ "dialect": "Guess 2", "clarification": "Meaning 2", "tone": "Tone 2", "context": "Context 2", "pragmatics": "Intent 2" }},
            {{ "dialect": "Guess 3", "clarification": "Meaning 3", "tone": "Tone 3", "context": "Context 3", "pragmatics": "Intent 3" }}
        ]
        """
        try:
            # UPGRADE: Use smart_generate for deep analysis
            response = self.gemini_manager.generate_smart(prompt)
            clean_text = re.sub(r"```json|```", "", response.text).strip()
            return json.loads(clean_text)
        except Exception as e:
            print(f"Fallback Gen Error: {e}")
            return [{
                "dialect": "Unknown",
                "clarification": "Could not generate hypothesis",
                "tone": "---",
                "context": "---",
                "pragmatics": f"Error: {str(e)[:50]}..."
            }]

    def detect_and_analyze(self, text, threshold=80):
        results = []

        # 1. DATABASE LOOKUP
        if self.lookup_list and text:
            matches = process.extract(text, self.lookup_list, scorer=fuzz.ratio, limit=3)

            for best_utterance, score, index in matches:
                if score >= threshold:
                    # MATCH FOUND IN CSV
                    if index < len(self.df):
                        row = self.df.iloc[index]

                        # Check if Pragmatic Analysis exists; if not, generate it (with Pro-Boost).
                        prag_analysis = row.get("Pragmatic_Analysis", "")
                        if pd.isna(prag_analysis) or str(prag_analysis).strip() in ["", "---", "nan"]:
                            prag_analysis = self.generate_single_pragmatics(text, row["Dialect"], row.get("Tone_Category", "General"))

                        results.append({
                            "Source": "🗄️ Database",
                            "Dialect": row["Dialect"],
                            "Clarification": row["Clarification"],
                            "Tone": row.get("Tone_Category", "---"),
                            "Context": row.get("Linguistic_Context", "---"),
                            "Pragmatic Analysis": prag_analysis
                        })
                    # MATCH FOUND IN JSON JARGON
                    else:
                        jargon_def = self.lab_profile["jargon"].get(best_utterance, "Defined in Codebook")
                        profile_dialect = self.lab_profile.get("lab_name", "Custom Profile")
                        results.append({
                            "Source": "📒 Codebook",
                            "Dialect": profile_dialect,
                            "Clarification": jargon_def,
                            "Tone": "Contextual",
                            "Context": "Found in Active Persona Codebook",
                            "Pragmatic Analysis": "Defined in Lab Profile"
                        })

        # 2. GENERATIVE FALLBACK (With Pro-Boost)
        if not results:
            print(f"🤔 Unknown phrase '{text}'. Triggering Gemini Fallback...")
            ai_guesses = self.generate_unknown_analysis(text)
            for guess in ai_guesses:
                results.append({
                    "Source": "✨ AI Generated",
                    "Dialect": guess.get("dialect", "Unknown"),
                    "Clarification": guess.get("clarification", "---"),
                    "Tone": guess.get("tone", "---"),
                    "Context": guess.get("context", "---"),
                    "Pragmatic Analysis": guess.get("pragmatics", "Hypothesis")
                })

        return results[:3]

    # --- RICH SUGGESTION ENGINE (Uses Pro-Boost) ---
    def get_rich_suggestions(self, text, dialect):
        if not self.gemini_manager or not text or not dialect: return []
        profile_context = json.dumps(self.lab_profile, indent=2)
        prompt = f"""
        interpret this {dialect} sentence: "{text}" using profile: {profile_context}
        Output 3 JSON options: [{{ "clarification": "", "tone": "", "context": "", "pragmatics": "" }}]
        """
        try:
            # UPGRADE: Use smart_generate
            response = self.gemini_manager.generate_smart(prompt)
            clean_text = re.sub(r"```json|```", "", response.text).strip()
            return json.loads(clean_text)
        except: return []

    # --- SYNTAX GENERATOR (Uses Fast/Flash Only) ---
    def generate_syntax_pattern(self, utterance):
        if not self.gemini_manager: return r"\b" + utterance.lower().replace(" ", r"\s?") + r"\b"
        prompt = f"Create Python Regex for: '{utterance}'. Output ONLY regex."
        try:
            # UPGRADE: Use fast_generate (Flash is enough for regex)
            return self.gemini_manager.generate_fast(prompt).text.strip()
        except: return r"\b" + utterance.lower().replace(" ", r"\s?") + r"\b"

# --- AGENT 4: TRUST ---
class AgentTrust:
    def __init__(self):
        self.lock = Lock()
        print("🛡️ Agent 4 (Trust) Online.")

    def log_to_ipfs(self, data):
        if not PINATA_JWT: return "Local-Log-Only"
        headers = {"Authorization": f"Bearer {PINATA_JWT}"}
        try:
            res = requests.post("https://api.pinata.cloud/pinning/pinJSONToIPFS", headers=headers, json=data)
            return res.json().get("IpfsHash", "Error")
        except: return "IPFS_Fail"

    def process_feedback(self, action, original_text, dialect, clarification, tone, context, brain_agent, audio_path=None, pragmatics=""):
        timestamp = pd.Timestamp.now().isoformat()
        feedback_data = {
            "original": original_text, "dialect": dialect, "clarification": clarification,
            "tone": tone, "linguistic_context": context, "pragmatics": pragmatics, "action": action, "timestamp": timestamp
        }
        self.log_to_ipfs(feedback_data)

        if action == "Suggest Update":
            syntax = brain_agent.generate_syntax_pattern(original_text)
            update_msg = self.update_dataset_csv(dialect, original_text, clarification, tone, context, syntax, audio_path, pragmatics)
            brain_agent.refresh_knowledge_base()
            return f"{update_msg}\n🤖 Syntax: {syntax}"
        return "Feedback Logged."

    def update_dataset_csv(self, dialect, utterance, clarification, tone, context, syntax, audio_path=None, pragmatics=""):
        clean_dialect = dialect.strip().title()
        if not clean_dialect.endswith("English") and not clean_dialect.endswith("Dialect"): clean_dialect += " Dialect"
        filepath = os.path.join(DATASET_DIR, f"{clean_dialect}.csv")

        with self.lock:
            if not os.path.exists(filepath):
                new_df = pd.DataFrame(columns=["Utterance", "Dialect", "Clarification", "Tone_Category", "Linguistic_Context", "Syntax_Pattern", "Pragmatic_Analysis", "file_name"])
                new_df.to_csv(filepath, index=False)

            df = pd.read_csv(filepath)
            if "Pragmatic_Analysis" not in df.columns: df["Pragmatic_Analysis"] = "---"
            for col in ["Tone_Category", "Linguistic_Context", "file_name", "Syntax_Pattern"]:
                if col not in df.columns: df[col] = "---"

            audio_ref = ""
            if audio_path and os.path.exists(audio_path):
                ext = os.path.splitext(audio_path)[1]
                unique_name = f"{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}_{random.randint(1000,9999)}{ext}"
                new_path = os.path.join(os.path.dirname(audio_path), unique_name)
                try:
                    shutil.copy2(audio_path, new_path)
                    audio_ref = hf_manager.upload_audio_sample(new_path, dialect)
                except Exception as e:
                    print(f"❌ Error copying audio: {e}")
                    audio_ref = "Error_Saving_Audio"

            # Create NEW row always
            new_row = pd.DataFrame([{
                "Utterance": utterance, "Dialect": clean_dialect, "Clarification": clarification,
                "Tone_Category": tone, "Linguistic_Context": context,
                "Pragmatic_Analysis": pragmatics,
                "Syntax_Pattern": syntax, "file_name": audio_ref
            }])
            df = pd.concat([df, new_row], ignore_index=True)
            msg = f"✅ Added Entry: '{utterance}' [{tone}]"

            df.to_csv(filepath, index=False)
            hf_manager.push_update(filepath, f"Update {clean_dialect}: {utterance}")
            return msg

# --- AGENT 3: UX ---
class AgentUX:
    def __init__(self, input_agent, brain_agent, trust_agent):
        self.input = input_agent
        self.brain = brain_agent
        self.trust = trust_agent
        self.last_audio_path = None
        self.suggestion_cache = {}
        print("🎨 Agent 3 (UX) Online: Building Interface...")

    def get_quota_status(self):
        if self.brain.gemini_manager: return self.brain.gemini_manager.get_status_string()
        return "Manager not active"

    def automated_pipeline(self, audio_path, language="en"):
        if not audio_path: return pd.DataFrame(), "Waiting for Input...", self.get_quota_status()

        self.last_audio_path = audio_path
        segments = self.input.transcribe(audio_path, language)
        results = []

        for seg in segments:
            raw = seg["text"]
            possible_interpretations = self.brain.detect_and_analyze(raw)
            for interp in possible_interpretations:
                results.append({
                    "Source": interp["Source"],
                    "Speaker": seg["speaker"],
                    "Utterance": raw,
                    "Dialect": interp["Dialect"],
                    "Clarification": interp["Clarification"],
                    "Tone": interp["Tone"],
                    "Context": interp["Context"],
                    "Pragmatic Analysis": interp["Pragmatic Analysis"]
                })

        return pd.DataFrame(results), "✅ Analysis Complete", self.get_quota_status()

    def launch(self):
        existing_dialects = []
        if os.path.exists(DATASET_DIR):
            csv_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
            existing_dialects = [os.path.basename(f).replace(".csv", "") for f in csv_files]
        dropdown_choices = existing_dialects + ["+ Add New Dialect"]

        with gr.Blocks(theme=gr.themes.Soft()) as ui:
            gr.Markdown("## 🌍 IEDI-M²: Active Listening & Dialect Mediator")

            with gr.Tabs():
                with gr.Tab("🎙️ Live Analysis"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            audio_input = gr.Audio(label="Step 1: Speak/Upload", sources=["microphone", "upload"], type="filepath")
                            lang_select = gr.Dropdown(["en", "ko", "fr"], value="en", label="Step 2: Language (Optional)")
                            analyze_btn = gr.Button("Re-Run Analysis 🔄", variant="secondary")
                            quota_display = gr.Textbox(label="📊 Model Status", value=self.get_quota_status(), interactive=False)

                        with gr.Column(scale=2):
                            status_box = gr.Textbox(label="Status", interactive=False)
                            results_df = gr.Dataframe(
                                headers=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"],
                                interactive=False, label="Analysis Results", type="pandas", wrap=True
                            )

                    gr.Markdown("### ✍️ Active Feedback Loop")
                    with gr.Row():
                        with gr.Column(scale=1):
                            orig_text_state = gr.Textbox(visible=True, label="Original Text")
                            with gr.Row():
                                dialect_dropdown = gr.Dropdown(choices=dropdown_choices, label="Select Dialect", interactive=True)
                                new_dialect_input = gr.Textbox(label="Enter New Dialect Name", visible=False, interactive=True)
                        with gr.Column(scale=1):
                            suggestion_dropdown = gr.Dropdown(label="Suggest Clarification", choices=[], allow_custom_value=True, interactive=True)
                            selected_tone_state = gr.Textbox(label="Linguistic Tone", interactive=True)
                            selected_context_state = gr.TextArea(label="Linguistic Context", interactive=True, lines=2)
                            selected_pragmatics_state = gr.TextArea(label="Pragmatic Analysis", interactive=True, lines=2)

                    with gr.Row():
                        btn_suggest = gr.Button("💾 Suggest Update", variant="primary")
                    feedback_out = gr.Markdown()

                with gr.Tab("⚙️ Lab Context"):
                    profile_editor = gr.Code(value=self.brain.get_profile_text, language="json", label="nsl_lab_profile.json", lines=20)
                    save_profile_btn = gr.Button("💾 Save & Reload Profile", variant="primary")
                    profile_status = gr.Textbox(label="System Response", interactive=False)

            # --- EVENT LOGIC ---
            def update_suggestions_rich(text, dialect):
                try:
                    if not text or not dialect or dialect == "+ Add New Dialect":
                        return gr.update(choices=[]), "", "", "", self.get_quota_status()
                    suggestions_data = self.brain.get_rich_suggestions(text, dialect)
                    self.suggestion_cache = {}
                    display_choices = []
                    if not suggestions_data: return gr.update(choices=["No suggestions"]), "", "", "", self.get_quota_status()
                    for item in suggestions_data:
                        clar, tone, ctx = item.get("clarification", ""), item.get("tone", ""), item.get("context", "")
                        prag = item.get("pragmatics", "Auto-generated")
                        display_str = f"{clar}  [{tone}]"
                        display_choices.append(display_str)
                        self.suggestion_cache[display_str] = {"clar": clar, "tone": tone, "context": ctx, "pragmatics": prag}
                    if display_choices:
                        first = self.suggestion_cache[display_choices[0]]
                        return gr.update(choices=display_choices, value=display_choices[0]), first["tone"], first["context"], first["pragmatics"], self.get_quota_status()
                    return gr.update(choices=[]), "", "", "", self.get_quota_status()
                except: return gr.update(choices=["Error"]), "Error", "", "", self.get_quota_status()

            def on_suggestion_select(val):
                if val in self.suggestion_cache:
                    return self.suggestion_cache[val]["tone"], self.suggestion_cache[val]["context"], self.suggestion_cache[val]["pragmatics"]
                return "Custom", "User provided", ""

            dialect_dropdown.change(fn=update_suggestions_rich, inputs=[orig_text_state, dialect_dropdown], outputs=[suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state, quota_display])
            orig_text_state.blur(fn=update_suggestions_rich, inputs=[orig_text_state, dialect_dropdown], outputs=[suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state, quota_display])
            suggestion_dropdown.change(fn=on_suggestion_select, inputs=[suggestion_dropdown], outputs=[selected_tone_state, selected_context_state, selected_pragmatics_state])

            audio_input.change(self.automated_pipeline, [audio_input, lang_select], [results_df, status_box, quota_display])
            analyze_btn.click(self.automated_pipeline, [audio_input, lang_select], [results_df, status_box, quota_display])

            def handle_selection(evt: gr.SelectData, df):
                if df is None or len(df) == 0: return "", "", "", "", "", ""
                try:
                    row = df.iloc[evt.index[0]]
                    d = row["Dialect"] if row["Dialect"] in existing_dialects else None
                    return row["Utterance"], d, row["Clarification"], row["Tone"], row["Context"], row["Pragmatic Analysis"]
                except: return "", "", "", "", "", ""
            results_df.select(handle_selection, [results_df], [orig_text_state, dialect_dropdown, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state])

            def submit_logic(action, orig, d_drop, d_new, clar_raw, tone, context, prag):
                try:
                    final_d = d_new.strip() if d_drop == "+ Add New Dialect" else d_drop
                    if not final_d or not orig: return "❌ Invalid Input"
                    final_clar = str(clar_raw).rsplit("[", 1)[0].strip() if "[" in str(clar_raw) else clar_raw
                    audio_ref = self.last_audio_path if action == "Suggest Update" else None
                    return self.trust.process_feedback(action, orig, final_d, final_clar, tone, context, self.brain, audio_ref, prag)
                except Exception as e: return f"❌ Error: {e}"

            btn_suggest.click(lambda o, d, n, c, t, ctx, p: submit_logic("Suggest Update", o, d, n, c, t, ctx, p), [orig_text_state, dialect_dropdown, new_dialect_input, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state], [feedback_out])

            def on_dialect_change(val): return gr.update(visible=True) if val == "+ Add New Dialect" else gr.update(visible=False)
            dialect_dropdown.change(on_dialect_change, inputs=dialect_dropdown, outputs=new_dialect_input)
            save_profile_btn.click(self.brain.save_lab_profile, inputs=[profile_editor], outputs=[profile_status])

        ui.launch(share=True, debug=True)

# --- START SYSTEM ---
agent1 = AgentInput()
agent2 = AgentInterpretation(gemini_manager)
agent4 = AgentTrust()
agent3 = AgentUX(agent1, agent2, agent4)
agent3.launch()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/de

🧠 Gemini Manager Online: Flash-First Mode with Pro-Boost.
⬇️ Pulling datasets from Hugging Face...


American%20English.csv:   0%|          | 0.00/6.12k [00:00<?, ?B/s]

Indian%20English.csv:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  .../20260120_084228_8830.wav: 100%|##########| 90.9kB / 90.9kB            

⬆️ Pushing update: American English.csv...
✅ Sync Complete!


⚠️ Pro-Boost Failed (404 POST https://generativelanguage.googleapis.com...). Falling back to Flash.


⚠️ Pro-Boost Failed (404 POST https://generativelanguage.googleapis.com...). Falling back to Flash.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9ed0d0cc248c0125e2.gradio.live


## 🚀 Getting Started with Hardhat

Hardhat is a development environment for compiling, deploying, testing, and debugging your Ethereum software. It helps developers manage and automate the recurring tasks that are inherent to building smart contracts and dApps.

### 1. Install Node.js and npm (if you don't have them)
Hardhat projects are typically set up using Node.js and its package manager, `npm`. You can download Node.js (which includes npm) from the official website: [nodejs.org](https://nodejs.org/en/download/).

### 2. Create a New Project Directory
It's best to create a dedicated directory for your Hardhat project.


In [ ]:
import os

project_name = "my-hardhat-project"
if not os.path.exists(project_name):
    os.makedirs(project_name)
    print(f"Created directory: {project_name}")
else:
    print(f"Directory '{project_name}' already exists.")

# Change to the new directory
%cd {project_name}

### 3. Initialize the Project and Install Hardhat

Inside your project directory, you'll initialize a new npm project and then install Hardhat locally.


In [ ]:
!npm init -y
!npm install --save-dev hardhat

### 4. Create a Hardhat Project

Now you can run the Hardhat command to create your first project. It will ask you to choose a project type (e.g., "Create a basic sample project"). You can select the default options.


In [ ]:
!npx hardhat

After running `npx hardhat`, you'll have a basic project structure with sample contracts, scripts, and tests. You can explore these files in the file browser (`/content/my-hardhat-project`).

### Next Steps:
*   **Explore `hardhat.config.js`**: This is where you configure your network, compilers, and plugins.
*   **Write Smart Contracts**: Look into the `contracts/` directory to start writing your Solidity code.
*   **Write Tests**: Use the `test/` directory to write tests for your contracts.
*   **Run Scripts**: The `scripts/` directory is for deployment and interaction scripts.

Let me know if you want to compile, deploy, or interact with a sample contract!

In [ ]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Load keys
load_dotenv()
PINATA_JWT = os.getenv("PINATA_JWT")

def fetch_ipfs_logs():
    if not PINATA_JWT:
        print("❌ Error: PINATA_JWT not found.")
        return

    print("🔍 Fetching pinned files from Pinata...")

    url = "https://api.pinata.cloud/data/pinList?status=pinned"
    headers = {"Authorization": f"Bearer {PINATA_JWT}"}

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        files = response.json().get('rows', [])

        print(f"✅ Found {len(files)} pinned logs.")

        all_logs = []

        for file in files:
            cid = file['ipfs_pin_hash']
            # Fetch content from a public gateway
            gateway_url = f"https://gateway.pinata.cloud/ipfs/{cid}"
            try:
                log_data = requests.get(gateway_url).json()
                # Add CID for reference
                log_data['ipfs_cid'] = cid
                all_logs.append(log_data)
                print(f"   -> Retrieved log: {cid}")
            except Exception as e:
                print(f"   ⚠️ Could not read content for {cid}: {e}")

        # Convert to DataFrame for easy viewing
        if all_logs:
            df = pd.DataFrame(all_logs)
            print("\n📊 Retrieved Data Summary:")
            print(df.head())

            # Save to CSV for analysis
            df.to_csv("ipfs_audit_trail.csv", index=False)
            print("\n💾 Saved full log to 'ipfs_audit_trail.csv'")
            return df
        else:
            print("No valid logs found.")

    except Exception as e:
        print(f"❌ API Error: {e}")

# Run the retrieval
audit_df = fetch_ipfs_logs()

❌ Error: PINATA_JWT not found.
